# Retail Demand Forecasting — Modeling Prep: Cleaned Development Dataset

This notebook picks up where `retail_demand_eda.ipynb` left off and produces a
**cleaned, model-ready development dataset**. It:

1. Rebuilds the wrangled + feature-engineered `df` from the EDA notebook (Sections 1.1–1.4)
2. Reviews the two dataset questions below against the *actual* data
3. Creates dummy/indicator features for every categorical variable
4. Standardizes numeric features with a scaler (fit on train, applied to test)
5. Splits into training and testing sets (time-based, consistent with the EDA notebook)
6. Saves the cleaned `train` / `test` sets to CSV for the modeling step

**Review questions**
- *Does my dataset have any categorical data, such as Gender or day of the week?*
- *Do my features have data values that range from 0–100, 0–1, or both — and more?*

Both are answered explicitly in Section 2 below, using the real column dtypes and
ranges rather than assuming the answer.

## 0. Setup

In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
#Make all variables & outputs from the EDA steps performed in notebook : retail_demand_eda.ipynb
import papermill as pm
pm.execute_notebook("retail_demand_eda.ipynb", "retail_demand_modeling.ipynb", parameters=dict())

pd.set_option("display.max_columns", 60)
RANDOM_SEED = 42

Executing:   0%|          | 0/34 [00:00<?, ?cell/s]

## 1. Rebuild the wrangled dataset (from the EDA notebook)

Same synthetic loaders and feature engineering as `retail_demand_eda.ipynb`
Sections 1.1–1.4, so this notebook is a self-contained continuation.
When real data is available, swap `load_sales()` / `load_weather()` / `load_holidays()`
for real loaders — everything downstream is unchanged.

In [8]:
def load_sales(start="2019-01-01", end="2023-12-31", n_categories=6, n_stores=3, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, end, freq="D")
    categories = ["Beverages", "Produce", "Bakery", "Frozen Foods",
                  "Cold Weather Apparel", "Cleaning Supplies"][:n_categories]
    stores = [f"Store_{i+1}" for i in range(n_stores)]

    base_level = {"Beverages": 400, "Produce": 350, "Bakery": 200, "Frozen Foods": 180,
                  "Cold Weather Apparel": 60, "Cleaning Supplies": 120}
    weekly_amp = {"Beverages": 80, "Produce": 40, "Bakery": 60, "Frozen Foods": 20,
                  "Cold Weather Apparel": 10, "Cleaning Supplies": 15}

    rows = []
    for store in stores:
        store_factor = rng.uniform(0.85, 1.15)
        for cat in categories:
            level = base_level[cat] * store_factor
            noise = rng.normal(0, level * 0.08, size=len(dates))
            dow = dates.dayofweek.values
            weekly = weekly_amp[cat] * np.where(dow >= 5, 1, -0.3)
            annual = level * 0.15 * np.sin(2 * np.pi * (dates.dayofyear.values / 365.25))
            trend = np.linspace(0, level * 0.1, len(dates))
            sales = level + weekly + annual + trend + noise
            sales = np.clip(sales, 0, None)
            rows.append(pd.DataFrame({"date": dates, "store": store, "category": cat,
                                       "sales": sales.round(0)}))
    df = pd.concat(rows, ignore_index=True)

    stockout_mask = rng.random(len(df)) < 0.01
    df.loc[stockout_mask, "sales"] = 0
    df["stockout_flag"] = stockout_mask

    df["on_promo"] = rng.random(len(df)) < 0.08
    df.loc[df["on_promo"], "sales"] *= rng.uniform(1.2, 1.6, size=df["on_promo"].sum())
    return df


def load_weather(start="2019-01-01", end="2023-12-31", seed=RANDOM_SEED):
    rng = np.random.default_rng(seed + 1)
    dates = pd.date_range(start, end, freq="D")
    doy = dates.dayofyear.values
    seasonal_temp = 55 + 25 * np.sin(2 * np.pi * (doy - 105) / 365.25)
    temp = seasonal_temp + rng.normal(0, 8, size=len(dates))
    precip = np.clip(rng.gamma(shape=0.6, scale=0.25, size=len(dates)), 0, None)
    precip[rng.random(len(dates)) > 0.35] = 0
    snow_flag = (temp < 34) & (precip > 0.05) & (rng.random(len(dates)) < 0.6)
    extreme_cold = temp < 20
    extreme_heat = temp > 92
    return pd.DataFrame({"date": dates, "temp_f": temp.round(1), "precip_in": precip.round(2),
        "snow_flag": snow_flag, "extreme_cold": extreme_cold, "extreme_heat": extreme_heat})


def load_holidays(start="2019-01-01", end="2023-12-31"):
    years = range(pd.Timestamp(start).year, pd.Timestamp(end).year + 1)
    fixed = [("01-01", "New Year\'s Day", "national"), ("07-04", "Independence Day", "national"),
             ("12-25", "Christmas Day", "national"), ("11-11", "Veterans Day", "national"),
             ("02-14", "Valentine\'s Day", "promotional"), ("10-31", "Halloween", "promotional")]
    rows = []
    for y in years:
        for md_, name, htype in fixed:
            rows.append({"date": pd.Timestamp(f"{y}-{md_}"), "holiday_name": name, "holiday_type": htype})
        nov = pd.date_range(f"{y}-11-01", f"{y}-11-30", freq="D")
        thursdays = nov[nov.dayofweek == 3]
        rows.append({"date": thursdays[3], "holiday_name": "Thanksgiving", "holiday_type": "national"})
    return pd.DataFrame(rows).drop_duplicates(subset="date").sort_values("date").reset_index(drop=True)


sales_df = load_sales()
weather_df = load_weather()
holidays_df = load_holidays()

df = sales_df.merge(weather_df, on="date", how="left")
df = df.merge(holidays_df[["date", "holiday_name", "holiday_type"]], on="date", how="left")
df["is_holiday"] = df["holiday_name"].notna()

df["temp_f"] = df["temp_f"].interpolate(limit_direction="both")
df["precip_in"] = df["precip_in"].fillna(0)

df = df.sort_values(["store", "category", "date"]).reset_index(drop=True)
df["dow"] = df["date"].dt.dayofweek
df["is_weekend"] = df["dow"] >= 5
df["month"] = df["date"].dt.month
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

grp = df.groupby(["store", "category"])["sales"]
df["sales_lag_1"] = grp.shift(1)
df["sales_lag_7"] = grp.shift(7)
df["sales_roll_7"] = grp.transform(lambda s: s.shift(1).rolling(7).mean())
df["sales_roll_28"] = grp.transform(lambda s: s.shift(1).rolling(28).mean())

hol_dates = pd.Series(sorted(holidays_df["date"].unique()))
def days_to_nearest_holiday(dates):
    dates = pd.to_datetime(dates)
    idx = np.searchsorted(hol_dates.values, dates.values)
    idx = np.clip(idx, 1, len(hol_dates) - 1)
    prev_gap = (dates.values - hol_dates.values[idx - 1]).astype("timedelta64[D]").astype(int)
    next_gap = (hol_dates.values[idx] - dates.values).astype("timedelta64[D]").astype(int)
    return np.minimum(prev_gap, next_gap)
df["days_to_holiday"] = days_to_nearest_holiday(df["date"])

df["weather_bucket"] = np.select(
    [df["extreme_cold"], df["extreme_heat"], df["snow_flag"], df["precip_in"] > 0],
    ["Extreme cold", "Extreme heat", "Snow", "Rain"], default="Clear/mild")

print(df.shape)
df.head()

(32868, 24)


,date,store,category,sales,stockout_flag,on_promo,temp_f,precip_in,snow_flag,extreme_cold,extreme_heat,holiday_name,holiday_type,is_holiday,dow,is_weekend,month,week_of_year,sales_lag_1,sales_lag_7,sales_roll_7,sales_roll_28,days_to_holiday,weather_bucket
0,2019-01-01,Store_1,Bakery,211.0,False,False,32.5,0.00,False,False,False,New Year's Day,national,True,1,False,1,1,NaN,NaN,NaN,NaN,0,Clear/mild
1,2019-01-02,Store_1,Bakery,210.0,False,False,35.9,0.18,False,False,False,NaN,NaN,False,2,False,1,1,211.0,NaN,NaN,NaN,1,Rain
2,2019-01-03,Store_1,Bakery,206.0,False,False,25.7,0.00,False,False,False,NaN,NaN,False,3,False,1,1,210.0,NaN,NaN,NaN,2,Clear/mild
3,2019-01-04,Store_1,Bakery,197.0,False,False,23.1,0.00,False,False,False,NaN,NaN,False,4,False,1,1,206.0,NaN,NaN,NaN,3,Clear/mild
4,2019-01-05,Store_1,Bakery,312.0,False,False,14.3,0.00,False,True,False,NaN,NaN,False,5,True,1,1,197.0,NaN,NaN,NaN,4,Extreme cold


### 1.1 One small cleaning fix

`days_to_holiday` should never be negative, but at a few year-boundary dates the
`datetime64[us]` vs `datetime64[ns]` subtraction produces a small negative value
(a unit-mismatch edge case, not a real "days before" concept). Clip it at 0.

In [9]:
print("Rows with negative days_to_holiday before fix:", (df["days_to_holiday"] < 0).sum())
df["days_to_holiday"] = df["days_to_holiday"].clip(lower=0)

Rows with negative days_to_holiday before fix: 108


## 2. Review questions

### 2.1 Does my dataset have any categorical data, such as Gender or day of the week?

**Yes.** Let's list every non-numeric / label-style column and its distinct values.

In [10]:
categorical_like_cols = []
for c in df.columns:
    if df[c].dtype == object or str(df[c].dtype) == "bool":
        categorical_like_cols.append(c)
    elif c in ["dow", "month"]:  # numeric but really category labels
        categorical_like_cols.append(c)

for c in categorical_like_cols:
    n_unique = df[c].nunique(dropna=False)
    print(f"{c:16s} dtype={str(df[c].dtype):8s} n_unique={n_unique:3d}  sample={df[c].unique()[:6]}")

store            dtype=object   n_unique=  3  sample=['Store_1' 'Store_2' 'Store_3']
category         dtype=object   n_unique=  6  sample=['Bakery' 'Beverages' 'Cleaning Supplies' 'Cold Weather Apparel'
 'Frozen Foods' 'Produce']
stockout_flag    dtype=bool     n_unique=  2  sample=[False  True]
on_promo         dtype=bool     n_unique=  2  sample=[False  True]
snow_flag        dtype=bool     n_unique=  2  sample=[False  True]
extreme_cold     dtype=bool     n_unique=  2  sample=[False  True]
extreme_heat     dtype=bool     n_unique=  2  sample=[False  True]
holiday_name     dtype=object   n_unique=  8  sample=["New Year's Day" nan "Valentine's Day" 'Independence Day' 'Halloween'
 'Veterans Day']
holiday_type     dtype=object   n_unique=  3  sample=['national' nan 'promotional']
is_holiday       dtype=bool     n_unique=  2  sample=[ True False]
dow              dtype=int32    n_unique=  7  sample=[1 2 3 4 5 6]
is_weekend       dtype=bool     n_unique=  2  sample=[False  True]
month    

This dataset doesn't have a `Gender` column, but it has the day-of-week
equivalent and more: `store`, `category`, `dow` (day of week, 0=Mon–6=Sun),
`month`, `holiday_type`, `weather_bucket`, plus a set of boolean indicator flags
(`is_holiday`, `is_weekend`, `on_promo`, `stockout_flag`, `snow_flag`,
`extreme_cold`, `extreme_heat`) that are already 0/1 but represent categorical
concepts. `store`, `category`, `dow`, `month`, `holiday_type`, and
`weather_bucket` are true multi-level categoricals that need dummy/indicator
encoding before modeling — that's Section 3 below.

### 2.2 Do my features have values ranging from 0–100, 0–1, or both — and more?

**Yes — the numeric features span very different scales.** This is exactly why
standardization is needed before feeding features into a scale-sensitive model
(e.g. linear/regularized regression, KNN, distance- or gradient-based methods).

In [11]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
ranges = df[num_cols].agg(["min", "max"]).T
ranges["span"] = ranges["max"] - ranges["min"]
ranges.sort_values("span")

,min,max,span
precip_in,0.000000,1.500000,1.500000
dow,0.000000,6.000000,6.000000
month,1.000000,12.000000,11.000000
week_of_year,1.000000,53.000000,52.000000
days_to_holiday,0.000000,70.000000,70.000000
temp_f,9.300000,102.200000,92.900000
sales_roll_28,46.428571,564.800150,518.371579
sales_roll_7,38.000000,629.670447,591.670447
sales_lag_7,0.000000,964.334376,964.334376
sales_lag_1,0.000000,964.334376,964.334376


Reading the table: boolean/indicator-style numeric columns sit on a **0–1**
scale (`dow` is 0–6, `month` is 1–12), `precip_in` is roughly **0–1.5**,
`temp_f` sits in a **~9–100+** range, and the sales-based features
(`sales`, `sales_lag_1`, `sales_lag_7`, `sales_roll_7`, `sales_roll_28`) run into
the **hundreds** (up to ~960+). So yes — the features have values in the 0–1
range, the 0–100 range, *and* well beyond both. Without standardizing, the
sales-scale features would dominate any distance- or gradient-based model
purely because of their magnitude, not their actual predictive signal.

## 3. Dummy / indicator features for categorical variables

- Drop `sales_lag_*` / `sales_roll_*` rows from the very start of each
  store-category series, where they're undefined (first 28 days of history).
- One-hot encode the multi-level categoricals: `store`, `category`, `dow`,
  `month`, `holiday_type` (filling missing = "None" for non-holiday days),
  and `weather_bucket`. `drop_first=True` avoids the dummy-variable trap.
- Cast the already-boolean indicator columns to 0/1 integers.
- Drop `holiday_name` (redundant with `holiday_type` / `is_holiday`) and keep
  `date` only for the train/test split, not as a model feature.

In [12]:
lag_cols = ["sales_lag_1", "sales_lag_7", "sales_roll_7", "sales_roll_28"]
before = len(df)
df_model = df.dropna(subset=lag_cols).copy()
print(f"Dropped {before - len(df_model)} rows with undefined lag/rolling features (series warm-up)")

df_model["holiday_type"] = df_model["holiday_type"].fillna("None")

cat_cols = ["store", "category", "dow", "month", "holiday_type", "weather_bucket"]
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

bool_cols = ["stockout_flag", "on_promo", "snow_flag", "extreme_cold",
             "extreme_heat", "is_holiday", "is_weekend"]
for c in bool_cols:
    df_encoded[c] = df_encoded[c].astype(int)

df_encoded = df_encoded.drop(columns=["holiday_name"])

print(df_encoded.shape)
df_encoded.head()

Dropped 504 rows with undefined lag/rolling features (series warm-up)
(32364, 47)


,date,sales,stockout_flag,on_promo,temp_f,precip_in,snow_flag,extreme_cold,extreme_heat,is_holiday,is_weekend,week_of_year,sales_lag_1,sales_lag_7,sales_roll_7,sales_roll_28,days_to_holiday,store_Store_2,store_Store_3,category_Beverages,category_Cleaning Supplies,category_Cold Weather Apparel,category_Frozen Foods,category_Produce,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,holiday_type_national,holiday_type_promotional,weather_bucket_Extreme cold,weather_bucket_Extreme heat,weather_bucket_Rain,weather_bucket_Snow
28,2019-01-29,321.766548,0,1,30.9,0.01,0,0,0,0,0,5,203.000000,286.784724,253.690266,232.279709,16,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
29,2019-01-30,217.000000,0,0,14.6,0.00,0,1,0,0,0,5,321.766548,245.000000,258.687669,236.235657,15,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
30,2019-01-31,305.346179,0,1,35.5,0.25,0,0,0,0,0,5,217.000000,246.047136,254.687669,236.485657,14,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
31,2019-02-01,198.000000,0,0,34.4,0.00,0,0,0,0,0,5,305.346179,223.000000,263.158961,240.033735,13,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
32,2019-02-02,299.000000,0,0,22.0,0.00,0,0,0,0,1,5,198.000000,279.000000,259.587532,240.069450,12,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 4. Standardize numeric features

Standardize (zero mean, unit variance) the continuous numeric predictors:
weather (`temp_f`, `precip_in`), calendar distance (`days_to_holiday`,
`week_of_year`), and the sales history features (`sales_lag_1`, `sales_lag_7`,
`sales_roll_7`, `sales_roll_28`).

The target (`sales`) is **left unscaled** — the goal is to forecast actual
sales units, not a standardized version of them — and the dummy/indicator
columns (already 0/1) are left as-is.

**The scaler is fit on the training set only** and then applied to the test
set, to avoid leaking information about the holdout period into training.

In [13]:
numeric_to_scale = ["temp_f", "precip_in", "days_to_holiday", "week_of_year",
                    "sales_lag_1", "sales_lag_7", "sales_roll_7", "sales_roll_28"]

print(df_encoded[numeric_to_scale + ["sales"]].agg(["min", "max"]).T)

                       min         max
temp_f            9.300000  102.200000
precip_in         0.000000    1.500000
days_to_holiday   0.000000   70.000000
week_of_year      1.000000   53.000000
sales_lag_1       0.000000  964.334376
sales_lag_7       0.000000  964.334376
sales_roll_7     38.000000  629.670447
sales_roll_28    46.428571  564.800150
sales             0.000000  964.334376


## 5. Train / test split

Time series data isn't shuffled — consistent with the EDA notebook (Section
1.5), the holdout ("test") set is the most recent 90 days for every
store/category series, and everything before that is "train".

In [14]:
cutoff = df_encoded["date"].max() - pd.Timedelta(days=90)
train = df_encoded[df_encoded["date"] <= cutoff].copy()
test = df_encoded[df_encoded["date"] > cutoff].copy()

print(f"Train: {train['date'].min().date()} to {train['date'].max().date()}  ({len(train)} rows)")
print(f"Test:  {test['date'].min().date()} to {test['date'].max().date()}  ({len(test)} rows)")

Train: 2019-01-29 to 2023-10-02  (30744 rows)
Test:  2023-10-03 to 2023-12-31  (1620 rows)


In [15]:
scaler = StandardScaler()
train[numeric_to_scale] = scaler.fit_transform(train[numeric_to_scale])
test[numeric_to_scale] = scaler.transform(test[numeric_to_scale])  # transform only, no fit

train[numeric_to_scale].describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
temp_f,1.756481e-17,1.000016,-2.446582,2.422573
precip_in,-6.927699e-17,1.000016,-0.367753,12.355864
days_to_holiday,-1.109356e-17,1.000016,-1.318350,2.170405
week_of_year,7.233928e-17,1.000016,-1.715882,1.855495
sales_lag_1,7.395709e-18,1.000016,-1.641669,4.953785
sales_lag_7,2.218713e-17,1.000016,-1.641409,4.951084
sales_roll_7,7.395709e-17,1.000016,-1.475117,2.844225
sales_roll_28,1.035399e-16,1.000016,-1.423447,2.384757


Train-set scaled features now have mean ≈ 0 / std ≈ 1 as expected. The test
set is transformed using the *training* scaler, so its mean/std won't be
exactly 0/1 (by design — it's held out data, not refit).

## 6. Save the cleaned development dataset

Write the cleaned, encoded, scaled `train` and `test` sets to CSV, ready for
the modeling step. `date` is kept as an identifier column (drop it, or keep it
as an index, at model-fit time — don't feed the raw timestamp to sklearn as-is).

In [16]:
train.to_csv("train_clean.csv", index=False)
test.to_csv("test_clean.csv", index=False)

print("Saved train_clean.csv:", train.shape)
print("Saved test_clean.csv: ", test.shape)
print("\nFinal feature columns:")
print([c for c in train.columns if c not in ("date", "sales")])

Saved train_clean.csv: (30744, 47)
Saved test_clean.csv:  (1620, 47)

Final feature columns:
['stockout_flag', 'on_promo', 'temp_f', 'precip_in', 'snow_flag', 'extreme_cold', 'extreme_heat', 'is_holiday', 'is_weekend', 'week_of_year', 'sales_lag_1', 'sales_lag_7', 'sales_roll_7', 'sales_roll_28', 'days_to_holiday', 'store_Store_2', 'store_Store_3', 'category_Beverages', 'category_Cleaning Supplies', 'category_Cold Weather Apparel', 'category_Frozen Foods', 'category_Produce', 'dow_1', 'dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12', 'holiday_type_national', 'holiday_type_promotional', 'weather_bucket_Extreme cold', 'weather_bucket_Extreme heat', 'weather_bucket_Rain', 'weather_bucket_Snow']


## 7. Summary

- **Categorical data confirmed:** `store`, `category`, day-of-week (`dow`),
  `month`, `holiday_type`, and `weather_bucket` were all one-hot/dummy encoded;
  existing boolean flags were cast to 0/1 indicators.
- **Magnitude confirmed to vary widely:** raw features ranged from 0–1
  (indicators, precipitation) to 0–100 (temperature, day/month) to the
  hundreds (sales and its lag/rolling features) — `StandardScaler` was fit on
  train and applied to both train and test so every numeric feature is on a
  comparable scale.
- **Split:** a time-respecting 90-day holdout, matching the EDA notebook, so
  no future information leaks into training.
- **Output:** `train_clean.csv` and `test_clean.csv` are the cleaned
  development dataset, ready to feed into the modeling step (e.g. Ridge/Lasso,
  gradient-boosted trees, or a seasonal baseline for comparison).